In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Results Visualization and Analysis\n",
    "## Comprehensive Analysis of Security Framework Performance\n",
    "\n",
    "This notebook:\n",
    "- Visualizes overall framework performance\n",
    "- Analyzes detection effectiveness per attack type\n",
    "- Shows real-time performance metrics\n",
    "- Creates publication-ready figures"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import sys\n",
    "sys.path.append('../src')\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import json\n",
    "import pickle\n",
    "from pathlib import Path\n",
    "\n",
    "# Set style\n",
    "sns.set_style('whitegrid')\n",
    "plt.rcParams['figure.figsize'] = (15, 6)\n",
    "plt.rcParams['font.size'] = 11\n",
    "\n",
    "print(\"✓ Libraries imported successfully\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load Results"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load model comparison results\n",
    "model_comparison = pd.read_csv('../results/model_comparison.csv')\n",
    "\n",
    "# Load dataset summary\n",
    "dataset_summary = pd.read_csv('../results/dataset_summary.csv')\n",
    "\n",
    "# Load model metadata\n",
    "with open('../results/model_metadata.json', 'r') as f:\n",
    "    metadata = json.load(f)\n",
    "\n",
    "print(\"✓ Results loaded\")\n",
    "print(f\"\\nBest Model: {metadata['best_model']}\")\n",
    "print(f\"Training Date: {metadata['training_date']}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Overall Framework Performance"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create comprehensive performance dashboard\n",
    "fig = plt.figure(figsize=(18, 10))\n",
    "gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)\n",
    "\n",
    "# Title\n",
    "fig.suptitle('CAN Bus Security Framework - Performance Dashboard', \n",
    "             fontsize=18, fontweight='bold', y=0.98)\n",
    "\n",
    "# 1. Model Comparison (Top Left - spans 2 columns)\n",
    "ax1 = fig.add_subplot(gs[0, :2])\n",
    "metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']\n",
    "x = np.arange(len(model_comparison))\n",
    "width = 0.2\n",
    "\n",
    "colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']\n",
    "for i, (metric, color) in enumerate(zip(metrics, colors)):\n",
    "    ax1.bar(x + i*width, model_comparison[metric], width, label=metric, color=color, alpha=0.8)\n",
    "\n",
    "ax1.set_xlabel('Model', fontsize=12, fontweight='bold')\n",
    "ax1.set_ylabel('Score', fontsize=12, fontweight='bold')\n",
    "ax1.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')\n",
    "ax1.set_xticks(x + width * 1.5)\n",
    "ax1.set_xticklabels(model_comparison['Model'])\n",
    "ax1.legend(loc='lower right')\n",
    "ax1.grid(axis='y', alpha=0.3)\n",
    "ax1.set_ylim([0, 1.05])\n",
    "\n",
    "# 2. Error Rates (Top Right)\n",
    "ax2 = fig.add_subplot(gs[0, 2])\n",
    "error_data = model_comparison[['Model', 'FPR', 'FNR']]\n",
    "x_err = np.arange(len(error_data))\n",
    "ax2.bar(x_err - 0.2, error_data['FPR'], 0.4, label='False Positive Rate', color='#e74c3c', alpha=0.8)\n",
    "ax2.bar(x_err + 0.2, error_data['FNR'], 0.4, label='False Negative Rate', color='#f39c12', alpha=0.8)\n",
    "ax2.set_xticks(x_err)\n",
    "ax2.set_xticklabels(error_data['Model'], rotation=15, ha='right')\n",
    "ax2.set_ylabel('Rate', fontsize=12, fontweight='bold')\n",
    "ax2.set_title('Error Rates', fontsize=14, fontweight='bold')\n",
    "ax2.legend()\n",
    "ax2.grid(axis='y', alpha=0.3)\n",
    "\n",
    "# 3. Dataset Size Comparison (Middle Left)\n",
    "ax3 = fig.add_subplot(gs[1, 0])\n",
    "dataset_sizes = dataset_summary['Total Messages']\n",
    "colors_ds = ['#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']\n",
    "ax3.pie(dataset_sizes, labels=dataset_summary['Dataset'], autopct='%1.1f%%',\n",
    "        startangle=90, colors=colors_ds)\n",
    "ax3.set_title('Dataset Distribution', fontsize=14, fontweight='bold')\n",
    "\n",
    "# 4. Message Rate Comparison (Middle Center)\n",
    "ax4 = fig.add_subplot(gs[1, 1])\n",
    "ax4.barh(dataset_summary['Dataset'], dataset_summary['Avg Rate (msg/s)'], \n",
    "         color=colors_ds, alpha=0.8)\n",
    "ax4.set_xlabel('Messages per Second', fontsize=12, fontweight='bold')\n",
    "ax4.set_title('Average Message Rates', fontsize=14, fontweight='bold')\n",
    "ax4.grid(axis='x', alpha=0.3)\n",
    "\n",
    "# 5. Data Entropy Comparison (Middle Right)\n",
    "ax5 = fig.add_subplot(gs[1, 2])\n",
    "ax5.bar(range(len(dataset_summary)), dataset_summary['Data Entropy (bits)'],\n",
    "        color=colors_ds, alpha=0.8)\n",
    "ax5.set_xticks(range(len(dataset_summary)))\n",
    "ax5.set_xticklabels(dataset_summary['Dataset'], rotation=15, ha='right')\n",
    "ax5.set_ylabel('Entropy (bits)', fontsize=12, fontweight='bold')\n",
    "ax5.set_title('Data Randomness', fontsize=14, fontweight='bold')\n",
    "ax5.grid(axis='y', alpha=0.3)\n",
    "\n",
    "# 6. Best Model Metrics (Bottom - spans all columns)\n",
    "ax6 = fig.add_subplot(gs[2, :])\n",
    "best_model_row = model_comparison.iloc[